Arrays · Study Notes  
Ref: EPI Chapter 2  

KTH  
June 30, 2026

## 1. Array fundamentals

The **array** is the simplest data structure — a contiguous block of memory,
usually used to represent sequences. For an array `A`, `A[i]` denotes the
`(i + 1)`-th object stored in the array.

| Operation | Time complexity |
|---|---|
| Retrieve / update `A[i]` | `O(1)` |
| Insert into a full array (with resizing) | `O(1)` amortized |
| Delete element at index `i` (length `n`) | `O(n - i)` |
| Insert a new element at index `i` | `O(n - i)` |

**Insertion & resizing.** Inserting into a *full* array is handled by
*resizing*: allocate a larger array and copy the entries over. This raises the
worst-case insertion cost, but if the new array is a constant factor larger than
the old one, resizing is infrequent and the **average** insertion cost stays
constant.

**Deletion.** Deleting an element requires shifting every successive element one
position to the left to fill the gap. For example, deleting the element at index
`4` from ⟨2, 3, 5, 7, 9, 11, 13, 17⟩ yields ⟨2, 3, 5, 7, 11, 13, 17, 0⟩ (the last
value doesn't matter).

## 2. Array set-up

**Problem.** Given an array of integers, reorder its entries so that the **even**
entries appear first. Solving it with `O(n)` extra space is easy — the challenge
is to do it **in place**.

**Key idea.** When working with arrays, take advantage of the fact that you can
operate efficiently on *both ends*. Partition the array into three subarrays that
appear in this order: **Even**, **Unclassified**, **Odd**. Initially Even and Odd
are empty and Unclassified is the whole array. Iterate through Unclassified,
swapping each element to the boundary of Even or Odd, thereby growing Even/Odd and
shrinking Unclassified.

In [1]:
def even_odd(A):
    next_even, next_odd = 0, len(A) - 1
    while next_even < next_odd:
        if A[next_even] % 2 == 0:
            # Already even: it's in the correct region, advance the left boundary.
            next_even += 1
        else:
            # Odd: swap it to the Odd region at the right end.
            A[next_even], A[next_odd] = A[next_odd], A[next_even]
            next_odd -= 1
    return A

In [2]:
# Quick demo
print(even_odd([3, 7, 1, 2, 4, 6, 5, 8]))

[8, 6, 4, 2, 1, 5, 7, 3]


**Complexity.** Space is `O(1)` — just a couple of index variables and a
temporary for swapping. We do a constant amount of work per entry, so time is
`O(n)`.

## 3. Top tips for arrays

- Array problems often have simple **brute-force** solutions using `O(n)` space,
  but subtler solutions use the array itself to reduce space to `O(1)`.
- **Filling from the front is slow** — see if you can write values from the back
  instead.
- Instead of **deleting** an entry (which shifts everything to its right),
  consider **overwriting** it.
- When integers are **encoded by an array**, consider processing digits from the
  back; alternately, reverse the array so the least-significant digit is first.
- Be comfortable **writing code that operates on subarrays**.
- It's incredibly easy to make **off-by-one** errors — reading past the last
  element is a common and catastrophic mistake.
- Don't worry about preserving the array's **integrity** (sortedness, grouping
  equal entries, etc.) until it's time to return.
- An array is a great structure when you know the **distribution of elements** in
  advance. E.g., a Boolean array of length `W` neatly represents a subset of
  `{0, 1, …, W − 1}`. (To represent a subset of `{1, 2, …, n}`, allocate size
  `n + 1` to simplify indexing.)
- On **2D arrays**, use parallel logic for rows and columns.
- Sometimes it's easier to **simulate** the specification than to solve for it
  analytically — e.g., for spiral order, just compute the output from the start.

## 4. Know your array libraries

Arrays in Python are provided by the **`list`** type. (The `tuple` type is very
similar but **immutable**.) A list is **dynamically resized** — there's no bound
on how many values it can hold, and values can be inserted or deleted at arbitrary
locations.

### 4.1 Instantiating lists

In [3]:
print([3, 5, 7, 11])
print([1] + [0] * 10)
print(list(range(100))[:10], "...")   # list from a range
print([[1, 2, 4], [3, 5, 7, 9], [13]])  # a 2D array

[3, 5, 7, 11]
[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9] ...
[[1, 2, 4], [3, 5, 7, 9], [13]]


### 4.2 Basic operations

In [4]:
A = [3, 5, 7, 11]
print("len:", len(A))
A.append(42);   print("append:", A)
A.remove(5);    print("remove:", A)   # removes first occurrence of the value 5
A.insert(1, 28); print("insert:", A)  # insert 28 at index 1

# Membership test is O(n) in the length of the list
print("42 in A:", 42 in A)

len: 4
append: [3, 5, 7, 11, 42]
remove: [3, 7, 11, 42]
insert: [3, 28, 7, 11, 42]
42 in A: True


### 4.3 Copy semantics

Understand the difference between `B = A` (both names refer to the *same* list)
and `B = list(A)` (a new list). Also know **shallow** vs **deep** copy —
`copy.copy(A)` vs `copy.deepcopy(A)`.

In [5]:
import copy

A = [1, 2, 3]
B = A          # alias: same object
C = list(A)    # shallow copy: new outer list
B.append(99)
print("A:", A, "| C:", C)   # A changed (B is A), C did not

nested = [[1, 2], [3, 4]]
shallow = copy.copy(nested)
deep = copy.deepcopy(nested)
nested[0].append(99)
print("shallow shares inner lists:", shallow)  # sees the 99
print("deep is fully independent:", deep)       # does not

A: [1, 2, 3, 99] | C: [1, 2, 3]
shallow shares inner lists: [[1, 2, 99], [3, 4]]
deep is fully independent: [[1, 2], [3, 4]]


### 4.4 Key list methods

`min(A)`, `max(A)`; binary search on **sorted** lists with
`bisect.bisect` / `bisect.bisect_left` / `bisect.bisect_right`;
`A.reverse()` (in place) vs `reversed(A)` (iterator);
`A.sort()` (in place) vs `sorted(A)` (returns a copy);
`del A[i]` (delete one element) and `del A[i:j]` (delete a slice).

In [6]:
import bisect

A = [1, 3, 3, 5, 7, 9]
print("min/max:", min(A), max(A))
print("bisect:", bisect.bisect(A, 6), "| left:", bisect.bisect_left(A, 3),
      "| right:", bisect.bisect_right(A, 3))

B = [5, 2, 9, 1]
print("sorted (copy):", sorted(B), "| original unchanged:", B)
B.sort();               print("sort (in place):", B)
print("reversed (iter):", list(reversed(B)))

C = [10, 20, 30, 40, 50]
del C[1]        # delete one element
del C[1:3]      # delete a slice
print("after del:", C)

min/max: 1 9
bisect: 4 | left: 1 | right: 3
sorted (copy): [1, 2, 5, 9] | original unchanged: [5, 2, 9, 1]
sort (in place): [1, 2, 5, 9]
reversed (iter): [9, 5, 2, 1]
after del: [10, 50]


### 4.5 Slicing

The most general slice is `A[i:j:k]`, with all of `i`, `j`, `k` optional. Slicing
can rotate a list (`A[k:] + A[:k]` rotates left by `k`) and make a shallow copy
(`B = A[:]`).

In [7]:
A = [1, 6, 3, 4, 5, 2, 7]
print("A[2:4]   =", A[2:4])
print("A[2:]    =", A[2:])
print("A[:4]    =", A[:4])
print("A[:-1]   =", A[:-1])
print("A[-3:]   =", A[-3:])
print("A[-3:-1] =", A[-3:-1])
print("A[1:5:2] =", A[1:5:2])
print("A[5:1:-2]=", A[5:1:-2])
print("A[::-1]  =", A[::-1])            # reverse
print("rotate 2 =", A[2:] + A[:2])     # rotate left by 2
print("copy     =", A[:])              # shallow copy

A[2:4]   = [3, 4]
A[2:]    = [3, 4, 5, 2, 7]
A[:4]    = [1, 6, 3, 4]
A[:-1]   = [1, 6, 3, 4, 5, 2]
A[-3:]   = [5, 2, 7]
A[-3:-1] = [5, 2]
A[1:5:2] = [6, 4]
A[5:1:-2]= [2, 4]
A[::-1]  = [7, 2, 5, 4, 3, 6, 1]
rotate 2 = [3, 4, 5, 2, 7, 1, 6]
copy     = [1, 6, 3, 4, 5, 2, 7]


### 4.6 List comprehensions

A list comprehension has four parts: an input sequence, an iterator over it, an
optional condition, and an expression producing each element. They're clearer than
`map()`/`filter()`/`lambda`. They support multiple loops (Cartesian products,
flattening a 2D list) and also work for **sets** and **dicts**.

> Rule of thumb: avoid more than two nested comprehensions — use conventional
> nested `for` loops instead, since the indentation is easier to read.

In [8]:
# Basic + conditional
print([x**2 for x in range(6)])
print([x**2 for x in range(6) if x % 2 == 0])

# Cartesian product of two sequences
A, B = [1, 3, 5], ['a', 'b']
print([(x, y) for x in A for y in B])

# Flatten a 2D list
M = [['a', 'b', 'c'], ['d', 'e', 'f']]
print([x for row in M for x in row])

# Per-entry transform of a 2D list
G = [[1, 2, 3], [4, 5, 6]]
print([[x**2 for x in row] for row in G])

[0, 1, 4, 9, 16, 25]
[0, 4, 16]
[(1, 'a'), (1, 'b'), (3, 'a'), (3, 'b'), (5, 'a'), (5, 'b')]
['a', 'b', 'c', 'd', 'e', 'f']
[[1, 4, 9], [16, 25, 36]]
